# **Data Preparation**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/aio26_train.csv')
test = pd.read_csv('/content/drive/MyDrive/aio26_test.csv')

In [ ]:
train.head()

,id,Followup_Days,Treatment_Assignment,Patient_Age_Days,Patient_Sex,Ascites_Indicator,Liver_Enlargement,Spider_Angioma,Edema_Status,Bilirubin_Level,Cholesterol_Level,Albumin_Level,Copper_Level,Alkaline_Phosphatase,AST_Level,Triglyceride_Level,Platelet_Count,Prothrombin_Time,Clinical_Stage,Status
0,7523,1416,Treatment_A,26612,Female,Absent,Absent,Absent,Controlled,0.50,NaN,3.517,15.1,726.3,83.97,NaN,258.2,9.87,2.0,C
1,10266,3182,Treatment_A,18647,Female,Absent,Absent,Absent,NaN,0.55,224.8,4.087,30.0,669.6,58.56,65.6,272.0,10.55,3.0,C
2,9705,1372,NaN,23408,Female,NaN,NaN,NaN,NaN,0.82,NaN,3.789,NaN,NaN,NaN,NaN,388.8,9.94,3.0,C
3,961,2077,NaN,27398,Female,NaN,NaN,NaN,NaN,0.90,NaN,3.095,NaN,NaN,NaN,NaN,NaN,10.50,4.0,C
4,3535,1121,NaN,18580,Female,NaN,NaN,NaN,NaN,0.80,NaN,3.093,NaN,NaN,NaN,NaN,189.4,10.85,3.0,D


In [ ]:
train = train.replace({None: "None"})
test = test.replace({None: "None"})

In [ ]:
train.shape, test.shape

((12000, 20), (10000, 19))

# **Handle Missing Values**

In [ ]:
# Count missing values in each column
missing_count = train.isnull().sum()

# Calculate missing percentage
missing_percent = train.isnull().mean() * 100

# Display the result
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing (%)": missing_percent
})

# Sort from highest to lowest missing percentage
missing_summary = missing_summary.sort_values(by="Missing (%)", ascending=False)

print(missing_summary)

                      Missing Count  Missing (%)
Triglyceride_Level             6695    55.791667
Cholesterol_Level              6658    55.483333
Copper_Level                   5256    43.800000
AST_Level                      5182    43.183333
Alkaline_Phosphatase           5179    43.158333
Platelet_Count                  445     3.708333
Prothrombin_Time                 14     0.116667
id                                0     0.000000
Patient_Age_Days                  0     0.000000
Treatment_Assignment              0     0.000000
Followup_Days                     0     0.000000
Patient_Sex                       0     0.000000
Albumin_Level                     0     0.000000
Bilirubin_Level                   0     0.000000
Edema_Status                      0     0.000000
Spider_Angioma                    0     0.000000
Ascites_Indicator                 0     0.000000
Liver_Enlargement                 0     0.000000
Clinical_Stage                    0     0.000000
Status              

In [ ]:
def handle_missing(df, threshold=50, drop_cols=None):

    # If drop_cols is not given, find columns to drop
    if drop_cols is None:
        missing_percent = df.isnull().mean() * 100
        drop_cols = []

        for col in df.columns:
            if missing_percent[col] > threshold:
                drop_cols.append(col)

    print("Columns to drop:", drop_cols)

    # Drop the selected columns
    df = df.drop(columns=drop_cols)

    # Fill missing values with the median
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            median = df[col].median()
            df[col] = df[col].fillna(median)

    # Reset the index
    df = df.reset_index(drop=True)

    return df, drop_cols

In [ ]:
train, drop_cols = handle_missing(train, threshold=50)
test, _ = handle_missing(test, drop_cols=drop_cols)

Columns to drop: ['Cholesterol_Level', 'Triglyceride_Level']
Columns to drop: ['Cholesterol_Level', 'Triglyceride_Level']


# **Encode Categorical Features**

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode features
for col in train.columns:
    if col != "Status" and train[col].dtype == "object":
        encoder = LabelEncoder()
        train[col] = encoder.fit_transform(train[col].astype(str))
        test[col] = encoder.transform(test[col].astype(str))

# Encode target
train["Status"] = LabelEncoder().fit_transform(train["Status"])

# **Train-Val Split**

In [ ]:
from sklearn.model_selection import train_test_split

train_data, valid_data = train_test_split(
    train,
    test_size=0.2,                  # 20% validation
    stratify=train["Status"],
    random_state=42,
    shuffle=True
)

train_data = train_data.reset_index(drop=True)
valid_data = valid_data.reset_index(drop=True)

In [ ]:
# Split features and labels
X_train = train_data.drop(columns=["Status"])
y_train = train_data["Status"]

X_valid = valid_data.drop(columns=["Status"])
y_valid = valid_data["Status"]

y_train.head()

,Status
0,2
1,2
2,2
3,2
4,2


# **Train with Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# Create the model
model = LogisticRegression(max_iter=5000)

# Train the model
model.fit(X_train, y_train)

# Predict probabilities
y_pred = model.predict_proba(X_valid)

# Calculate Log Loss
score = log_loss(y_valid, y_pred)

print("Validation Log Loss:", score)

Validation Log Loss: 0.5229034782781488


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# **Test & Submit**

In [ ]:
test_pred = model.predict_proba(test)

In [ ]:
submission_df = pd.read_csv('/content/drive/MyDrive/aio26_sample-submission.csv')
submission_df[['Status_C','Status_CL','Status_D']] = test_pred

In [ ]:
submission_df.to_csv('aio26_submission.csv', index=False)
print("Submission file saved as submission.csv!")

Submission file saved as submission.csv!
